## PHẦN 1: PHÉP KHỬ GAUSS VÀ CÁC ỨNG DỤNG
Mục tiêu của phần này là cài đặt thuật toán để giải quyết các bài toán cốt lõi của Đại số tuyến tính:
1. Giải hệ phương trình $Ax = b$ bằng phép khử Gauss có chọn phần tử chốt (Partial Pivoting).
2. Tìm Hạng ma trận và Cơ sở (không gian cột, không gian dòng, không gian nghiệm).
3. Tính Định thức và Ma trận nghịch đảo.

### 1. Khởi tạo hệ thống
Nạp các module tự cài đặt (`gaussian`, `determinant`, `inverse`, `rank_basis`) và module kiểm chứng (`verify`).

In [1]:
%load_ext autoreload
%autoreload 2

from gaussian import gaussian_elimination
from determinant import determinant
from inverse import inverse
from rank_basis import rank_and_basis
from verify import verify_solution, verify_determinant, verify_inverse, verify_rank_and_basis

### 2. Xây dựng bộ dữ liệu kiểm thử

In [2]:
# 1. Ma trận vuông, hệ có nghiệm duy nhất
A1 = [[ 1.0,  2.0, -1.0], 
      [ 2.0, -1.0,  1.0], 
      [ 3.0,  1.0, -2.0]]
b1 = [2.0, 3.0, -1.0]

# 2. Ma trận vuông cần hoán vị dòng (Số 0 ở pivot)
A2 = [[ 2.0,  4.0, -2.0], 
      [ 1.0,  2.0,  3.0], 
      [ 3.0, -1.0,  1.0]]
b2 = [-6.0, 5.0, 6.0]

# 3. Ma trận vuông, hệ vô nghiệm
A3 = [[ 1.0,  3.0, -2.0], 
      [ 2.0,  1.0,  4.0], 
      [ 0.0,  5.0, -8.0]]
b3 = [2.0, 7.0,  5.0]

# 4. Ma trận vuông, hệ vô số nghiệm
A4 = [[ 1.0,  3.0, -2.0], 
      [ 2.0,  1.0,  4.0], 
      [ 0.0,  5.0, -8.0]]
b4 = [2.0, 7.0, -3.0]

# 5. Ma trận chữ nhật
A5 = [[ 1.0, -1.0,  2.0,  1.0], 
      [ 2.0,  1.0, -1.0,  2.0], 
      [ 1.0,  2.0,  1.0, -1.0]]
b5 = [3.0, 4.0, 3.0]

# 6. Ma trận chữ nhật quá quyết
A6 = [[ 1.0,  2.0,  3.0], 
       [ 2.0,  5.0,  3.0], 
       [ 1.0,  0.0,  8.0],
       [ 4.0,  7.0, 14.0]]
b6 = [6.0, 10.0, 9.0, 25.0]

# 7. Ma trận vuông toàn số 0
A7 = [[0.0, 0.0, 0.0], 
       [0.0, 0.0, 0.0], 
       [0.0, 0.0, 0.0]]
b7 = [0.0, 0.0, 0.0]

# 8. Ma trận vuông có sai số rất nhỏ
A8 = [[ 1.0,  2.0, -1.0], 
      [ 2.0, -1.0,  3.0], 
      [ 3.0,  1.0,  2.00000000005]]
b8 = [4.0, 5.0, 9.0]

# 9. Ma trận vuông, gây ra ill-conditioned
A9 = [[ 5e-7,  0.0,   0.0,   0.0  ], 
      [ 0.0,   1.0,   0.0,   0.0  ], 
      [ 0.0,   0.0,   5e-7,  0.0  ],
      [ 0.0,   0.0,   0.0,   1.0  ]]
b9 = [1.0, 2.0, 3.0, 4.0]

# 10. Ma trận vuông nhưng vector b sai số chiều
A10 = [[ 1.0,  2.0,  3.0], 
      [ 4.0,  5.0,  6.0], 
      [ 7.0,  8.0,  9.0]]
b10 = [1.0, 2.0]

# 11. Ma trận rỗng
A11 = []
b11 = [0.0, 0.0, 0.0]

# 12. Ma trận có các dòng không cùng độ dài
A12 = [[ 1.0,  2.0,  3.0], 
       [ 4.0,  5.0],
       [ 7.0,  8.0,  9.0]]
b12 = [1.0, 2.0, 3.0]

### 3. Thực thi kiểm thử và đối chiếu
Vòng lặp dưới đây sẽ kiểm thử lần lượt 12 trường hợp. 
* Mã nguồn có sử dụng `try...except` để đảm bảo khi gặp dữ liệu lỗi, chương trình sẽ bắt lỗi mà không làm sập tiến trình.
* Mỗi bước tính toán (giải hệ, hạng, định thức, nghịch đảo) đều được đối chiếu với NumPy qua ngưỡng $\epsilon = 10^{-9}$.
* Các loại log kết quả:
    * **PASSED (Xanh lá):** Thuật toán chuẩn xác, sai số nằm trong ngưỡng cho phép và đúng với đáp án của NumPy (hoặc phản ánh đúng tính chất toán học của hệ).
    * **FAILED (Đỏ):** Thuật toán hoạt động sai lệch hoặc kết quả vượt quá ngưỡng sai số.
    * **ERROR (Vàng):** Thuật toán phát hiện và từ chối các đầu vào không hợp lệ (Exception Handling).

In [3]:
import copy

COLOR_FAILED = '\033[91m'
COLOR_PASSED = '\033[92m'
COLOR_WARNING = '\033[93m'
COLOR_RESET = '\033[0m'

test_cases = [
    ("TESTCASE 1: MA TRẬN VUÔNG, HỆ CÓ NGHIỆM", A1, b1),
    ("TESTCASE 2: MA TRẬN VUÔNG CẦN HOÁN VỊ DÒNG (SỐ 0 Ở PIVOT)", A2, b2),
    ("TESTCASE 3: MA TRẬN VUÔNG, HỆ VÔ NGHIỆM", A3, b3),
    ("TESTCASE 4: MA TRẬN VUÔNG, HỆ VÔ SỐ NGHIỆM", A4, b4),
    ("TESTCASE 5: MA TRẬN CHỮ NHẬT", A5, b5),
    ("TESTCASE 6: MA TRẬN CHỮ NHẬT QUÁ QUYẾT", A6, b6),
    ("TESTCASE 7: MA TRẬN VUÔNG TOÀN SỐ 0", A7, b7),
    ("TESTCASE 8: MA TRẬN VUÔNG CÓ SAI SỐ RẤT NHỎ", A8, b8),
    ("TESTCASE 9: MA TRẬN VUÔNG GÂY RA ILL-CONDITIONED", A9, b9),
    ("TESTCASE 10: MA TRẬN VUÔNG NHƯNG A VÀ b LỆCH CHIỀU", A10, b10),
    ("TESTCASE 11: MA TRẬN RỖNG", A11, b11),
    ("TESTCASE 12: MA TRẬN CÓ CÁC DÒNG KHÔNG CÙNG ĐỘ DÀI", A12, b12)
]

# Tiêu đề
for title, A_origin, b_origin in test_cases:
    print("\n" + "="*75)
    print(f"  {title}")
    print("="*75)

    # ----------------------------------------------------
    # [0] HIỂN THỊ DỮ LIỆU ĐẦU VÀO
    # ----------------------------------------------------
    print("[0] DỮ LIỆU ĐẦU VÀO:")
    print("   + Ma trận A:")
    if not A_origin or len(A_origin) == 0:
        print("     [] (Ma trận rỗng)")
    else:
        for row in A_origin:
            print(f"     {row}")
    print(f"   + Vector b: {b_origin}")
    print("-" * 30)
    
    # ----------------------------------------------------
    # [1] TEST GIẢI HỆ PHƯƠNG TRÌNH
    # ----------------------------------------------------
    print("\n[1] Kiểm tra giải hệ Ax = b:")
    try:
        U, x, swaps = gaussian_elimination(copy.deepcopy(A_origin), copy.deepcopy(b_origin))
        verify_solution(A_origin, x, b_origin) 
    except Exception as e:
        print(f"{COLOR_WARNING}[ERROR] Hàm xử lý từ chối giải: {e}{COLOR_RESET}")
    
    # ----------------------------------------------------
    # [2] TEST HẠNG & CƠ SỞ
    # ----------------------------------------------------
    print("\n[2] Kiểm tra hạng và cơ sở:")
    try:
        rank, col, row, null = rank_and_basis(copy.deepcopy(A_origin))
        verify_rank_and_basis(A_origin, rank, col, row, null)
    except Exception as e:
        print(f"{COLOR_WARNING}[ERROR] Hàm xử lý từ chối giải: {e}{COLOR_RESET}")
    
    # ----------------------------------------------------
    # [3] & [4] TEST ĐỊNH THỨC & NGHỊCH ĐẢO
    # ----------------------------------------------------
    m = len(A_origin)
    n = len(A_origin[0]) if m > 0 else 0
    
    # Chỉ chạy det/inv nếu ma trận VÀ vuông VÀ không rỗng
    if m == n and m > 0:
        print("\n[3] Kiểm tra định thức:")
        try:
            det = determinant(copy.deepcopy(A_origin))
            verify_determinant(A_origin, det)
        except Exception as e:
            print(f"{COLOR_WARNING}[ERROR] Lỗi hoặc từ chối tính: {e}{COLOR_RESET}")
            
        print("\n[4] Kiểm tra nghịch đảo:")
        try:
            inv = inverse(copy.deepcopy(A_origin))
            verify_inverse(A_origin, inv)
        except Exception as e:
            print(f"{COLOR_WARNING}[ERROR] Lỗi hoặc từ chối tính: {e}{COLOR_RESET}")
    else:
        print("\n[3] & [4] Bỏ qua kiểm tra định thức và nghịch đảo:")
        print(f"{COLOR_WARNING}>>> Lý do: Ma trận rỗng hoặc không phải ma trận vuông.{COLOR_RESET}")


  TESTCASE 1: MA TRẬN VUÔNG, HỆ CÓ NGHIỆM
[0] DỮ LIỆU ĐẦU VÀO:
   + Ma trận A:
     [1.0, 2.0, -1.0]
     [2.0, -1.0, 1.0]
     [3.0, 1.0, -2.0]
   + Vector b: [2.0, 3.0, -1.0]
------------------------------

[1] Kiểm tra giải hệ Ax = b:
   + Giá trị b gốc        : [ 2.  3. -1.]
   + Giá trị Ax thực tế   : [ 2.  3. -1.]
   + Nghiệm x bạn tính    : [1. 2. 3.]
   + Nghiệm x chuẩn NumPy : [1. 2. 3.]
PASSED: Ax = b (Sai số nằm trong ngưỡng cho phép).
PASSED: Vector nghiệm x khớp hoàn toàn với numpy.linalg.solve.


[2] Kiểm tra hạng và cơ sở:
   + Hạng bạn tính     : 3
   + Hạng chuẩn NumPy  : 3
PASSED: Hạng chuẩn xác.

   + Cơ sở cột bạn tính   : [[1.0, 2.0, 3.0], [2.0, -1.0, 1.0], [-1.0, 1.0, -2.0]]
   + Numpy tính toán      : Hạng gốc = 3, Hạng khi ghép cơ sở cột = 3
PASSED: Cơ sở không gian cột hợp lệ (cùng tập sinh với A).

   + Cơ sở dòng bạn tính  : [[1.0, 0.0, -0.6], [0.0, 1.0, -0.2], [0.0, 0.0, 2.0]]
   + NumPy tính toán      : Hạng gốc = 3, Hạng khi ghép cơ sở dòng = 3
PASSED: Cơ